# Artificial Neural Network
<br>

### Import packages

In [1]:
import numpy as np, pandas as pd, os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier

### My ANN code

In [2]:
class NeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.01, delta_j=0.00001):
        # Initialize weights
        self.W1 = np.random.randn(input_size, hidden_size)
        self.W2 = np.random.randn(hidden_size, output_size)
        self.learning_rate = learning_rate
        self.delta_j = delta_j

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))
    
    def forward(self, X):
        # Forward propagation
        self.z2 = np.dot(X, self.W1)
        self.a2 = self.sigmoid(self.z2)
        self.z3 = np.dot(self.a2, self.W2)
        self.a3 = self.sigmoid(self.z3)
        return self.a3
    
    def sigmoid_derivative(self, z):
        return z * (1 - z)
    
    def backprop(self, X, y):
        # Calculate output layer error
        output_error = self.a3 - y
        output_delta = output_error * self.sigmoid_derivative(self.a3)
        
        # Calculate hidden layer error
        hidden_error = np.dot(output_delta, self.W2.T)
        hidden_delta = hidden_error * self.sigmoid_derivative(self.a2)
        
        # Update weights
        self.W2 -= self.learning_rate * np.dot(self.a2.T, output_delta)
        self.W1 -= self.learning_rate * np.dot(X.T, hidden_delta)

    def cross_entropy_loss(self, y, y_pred):
        m = y.shape[0]
        y_pred = np.clip(y_pred, 1e-10, 1 - 1e-10) # Clip y_pred to prevent log(0) errors
        loss = -np.sum(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred)) / m
        return loss

    def train(self, X, y, iteration=100000):
        previous_cost = float('inf')
        for iteration in range(iteration):
            # Forward and backward propagation
            self.forward(X)
            self.backprop(X, y)
                
            # Compute cost
            current_cost = self.cross_entropy_loss(y, self.a3)
            
            # Check for convergence
            if abs(previous_cost - current_cost) < self.delta_j:
                print(f"Converged after {iteration + 1} iterations.")
                print(f"Cost function at convergence: {current_cost}")
                break
                
            previous_cost = current_cost

    def predict(self, X):
        output = self.forward(X)
        return np.round(output)

### Read data in (new dimensions from PCA)

In [3]:
df = pd.read_csv("PCA_new_dimensions.csv")
df

,PC1,PC2,PC3,PC4,PC5,PC6,Target,samples
0,-1.195869,34.675642,-22.566091,-10.124487,5.437070,-27.289066,0,sample_1
1,56.219885,13.707433,-5.176982,-17.500171,1.755895,-25.040687,1,sample_2
2,-9.829854,28.700385,-9.131298,-8.827665,-17.923073,-37.253751,0,sample_3
3,63.601822,7.700061,-6.719630,-26.422956,-22.782805,-22.051176,1,sample_4
4,-9.944483,32.471231,-27.308476,-18.768296,-26.321906,-25.460404,0,sample_5
...,...,...,...,...,...,...,...,...
858,27.400601,-41.364723,26.597015,-14.626021,14.450297,-21.922266,1,sample_860
859,-50.426578,-33.143071,26.495926,-5.602025,14.432004,-28.201209,0,sample_861
860,11.732831,-31.312210,13.967708,-13.244061,22.363992,-18.355105,1,sample_862
861,-45.341685,-25.790150,19.330514,-2.792202,24.088169,-22.854234,0,sample_863


### Subset dataframe and get the independent variables

In [4]:
X = df.drop(['Target', 'samples'], axis=1)
X

,PC1,PC2,PC3,PC4,PC5,PC6
0,-1.195869,34.675642,-22.566091,-10.124487,5.437070,-27.289066
1,56.219885,13.707433,-5.176982,-17.500171,1.755895,-25.040687
2,-9.829854,28.700385,-9.131298,-8.827665,-17.923073,-37.253751
3,63.601822,7.700061,-6.719630,-26.422956,-22.782805,-22.051176
4,-9.944483,32.471231,-27.308476,-18.768296,-26.321906,-25.460404
...,...,...,...,...,...,...
858,27.400601,-41.364723,26.597015,-14.626021,14.450297,-21.922266
859,-50.426578,-33.143071,26.495926,-5.602025,14.432004,-28.201209
860,11.732831,-31.312210,13.967708,-13.244061,22.363992,-18.355105
861,-45.341685,-25.790150,19.330514,-2.792202,24.088169,-22.854234


### Subset dataframe and get the target variable

In [5]:
y = df['Target']
y

0      0
1      1
2      0
3      1
4      0
      ..
858    1
859    0
860    1
861    0
862    1
Name: Target, Length: 863, dtype: int64

### Applying my ANN code

In [6]:
y = y.to_numpy()
y = y.reshape(-1, 1)  # reshape for binary classification

# Split data into 80% train and 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train custom neural network
nn = NeuralNetwork(input_size=6, hidden_size=3, output_size=1)
nn.train(X_train, y_train)

# Predictions and accuracy for custom model
y_pred_custom = nn.predict(X_test)
custom_accuracy = accuracy_score(y_test, y_pred_custom)
print(f"My model accuracy: {custom_accuracy:.2f}")

Converged after 379 iterations.
Cost function at convergence: 0.06621485454790067
My model accuracy: 0.98


### ANN with sklearn

In [7]:
# Two lines of code below is done again cause target variable does not need reshaping for sklearn
y = df['Target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

mlp = MLPClassifier(hidden_layer_sizes=(3,), max_iter=100000, random_state=42, tol=1e-6)
mlp.fit(X_train, y_train)
y_pred_sklearn = mlp.predict(X_test)

print("Number of iterations before convergence:", mlp.n_iter_)
print("Loss:", mlp.loss_)

sklearn_accuracy = accuracy_score(y_test, y_pred_sklearn)
print(f"Sklearn model accuracy: {sklearn_accuracy:.2f}")


Number of iterations before convergence: 2142
Loss: 0.03374353410323156
Sklearn model accuracy: 0.99


<br>
<br>

## Result summary:

### - The result when my code was used for the analysis was similar to the result gotten using the Sklearn package.
### - Both models (mine and SKlearn) had high accuracy scores indicating good model performance.